# CS521 HW3 - Problem 3

Jinwu Hu (jinwuhu2)

Completed implementation based on the [official starter notebook](https://github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw3/q3/multi_norm_adversarial_training_cifar10.ipynb). Use a GPU runtime. All code is embedded; the three public starter assets are downloaded below.

In [ ]:
%pip install -q "gdown==5.2.2"


In [ ]:
from pathlib import Path
import hashlib
import gdown
data = Path("/content/cs521_hw3/data")
data.mkdir(parents=True,exist_ok=True)
files = {
    "linf_base.pt": "17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes",
    "cifar10_train_10k.pt": "1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe",
    "cifar10_test_1k.pt": "1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA",
}
expected_sha256 = {
    "linf_base.pt": "ba1d32efdc1951d9b2c5a47a735e4a1701c9ae4a62e57ad6ace58af0d19405a8",
    "cifar10_train_10k.pt": "dac9de04f3bfc3fea36381fe62894b34f7b66f73de46f3ac2502e567f3816c00",
    "cifar10_test_1k.pt": "cb14f0bede10d8dd0ee36c1bc58f5d11e69d61ff9834918a6318b8f63411ce7a",
}
for name, file_id in files.items():
    if not (data/name).exists():
        gdown.download(id=file_id, output=str(data/name), quiet=False)
    actual = hashlib.sha256((data/name).read_bytes()).hexdigest()
    assert actual == expected_sha256[name], f"Unexpected data/checkpoint content: {name}"


## Setup, attacks, training, and evaluation

The default 10-epoch loop, training subset, radii, 50-step/two-restart evaluation, and supplied L1 attack are preserved. Timers and file logging are added. `pgd_linf`, `pgd_l2`, `single_norm_loss`, `ramp_loss`, `gradient_projection`, and `my_loss` are implemented.

In [ ]:
"""CS521 HW3 Q3: completed scaffold and reproducible experiment runner.
Adapted from the official nalinwadhwa02/CCS521FA26HW notebook.
"""
import argparse
import hashlib
import json
import os
from pathlib import Path
ROOT = Path("/content/cs521_hw3")
ROOT.mkdir(parents=True,exist_ok=True)
DATA_DIR = ROOT / "data"
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device(os.environ.get("CS521_DEVICE", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")))
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(DATA_DIR / path, map_location="cpu", weights_only=True)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
def initialize_data():
    global x_train, y_train, x_test, y_test
    x_train, y_train = load_data("cifar10_train_10k.pt")
    x_test, y_test = load_data("cifar10_test_1k.pt")
    print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape), flush=True)


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load(DATA_DIR / "linf_base.pt", map_location=device, weights_only=True))
    return model

def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = clip_box(x, (delta + alpha * grad.sign()).clamp(-eps, eps))
    return (x + delta).detach()

def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        # Uniform radius, matching the random-radius convention of the supplied L1 attack.
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * (eps * r)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1.0)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    history = []
    for epoch in range(epochs):
        sync_device()
        epoch_start = time.perf_counter()
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        sync_device()
        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.perf_counter() - epoch_start})
        print(json.dumps(history[-1]), flush=True)
    model.eval()
    model.training_history = history
    return model

def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)


def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q, logits_r = model(x_q), model(x_r)
    ce_q = F.cross_entropy(logits_q, y, reduction="none")
    ce_r = F.cross_entropy(logits_r, y, reduction="none")
    # PyTorch's target is p_q, input is log p_r: this is KL(p_q || p_r).
    kl = F.kl_div(F.log_softmax(logits_r, dim=1), F.softmax(logits_q, dim=1),
                  reduction="none").sum(dim=1)
    mask = (logits_q.argmax(1) == y).to(kl.dtype)
    # Mean over the qualifying examples; exactly zero when none qualify.
    paired = (kl * mask).sum() / mask.sum().clamp_min(1)
    return torch.maximum(ce_q, ce_r).mean() + lam * paired

def gradient_projection(g_n, g_a):
    cosine = (g_n.flatten() @ g_a.flatten()) / (g_n.norm() * g_a.norm()).clamp_min(1e-12)
    return cosine.clamp_min(0) * g_n


class AlternatingDistillationLoss:
    """Alternate L1/L2 updates while retaining an L-infinity-robust teacher.

    This is a student-method candidate, not a claim of global research novelty.
    Teacher targets never depend on test examples or test labels.
    """
    def __init__(self, teacher, distill_weight=0.5, clean_weight=0.25):
        self.teacher = teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.distill_weight = distill_weight
        self.clean_weight = clean_weight
        self.batch_index = 0
        self.stats = []

    def __call__(self, model, x, y):
        norm = ("l1", "l2")[self.batch_index % 2]
        self.batch_index += 1
        model.eval()
        x_alt = attack(model, x, y, norm)
        # Keep attack cost identical in the ablation to isolate distillation.
        x_anchor = attack(model, x, y, "linf")
        with torch.no_grad():
            teacher_logits = self.teacher(x)
            teacher_prob = F.softmax(teacher_logits, dim=1)
            reliable = (teacher_logits.argmax(1) == y)
        model.train()
        alt_logits = model(x_alt)
        anchor_logits = model(x_anchor)
        clean_logits = model(x)
        ce_alt = F.cross_entropy(alt_logits, y)
        ce_clean = F.cross_entropy(clean_logits, y)
        kd_each = F.kl_div(F.log_softmax(anchor_logits, dim=1), teacher_prob,
                          reduction="none").sum(dim=1)
        mask = reliable.to(kd_each.dtype)
        kd = (kd_each * mask).sum() / mask.sum().clamp_min(1)
        loss = ce_alt + self.distill_weight * kd + self.clean_weight * ce_clean
        self.stats.append({"norm": norm, "ce_alt": ce_alt.detach().item(),
                           "clean_ce": ce_clean.detach().item(), "distillation": kd.detach().item(),
                           "teacher_correct_fraction": mask.mean().item()})
        return loss

_my_loss_instance = None
def my_loss(model, x, y):
    global _my_loss_instance
    if _my_loss_instance is None:
        _my_loss_instance = AlternatingDistillationLoss(load_base_model())
    return _my_loss_instance(model, x, y)



def sync_device():
    if device.type == "cuda": torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()

def save_model(model, path):
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, path)

def execute_run(method, seed, epochs=10):
    dest = ROOT / "results/q3" / f"{method}_seed{seed}_epochs{epochs}"
    dest.mkdir(parents=True, exist_ok=True)
    result_path = dest / "result.json"
    if result_path.exists():
        print("SKIP_COMPLETE", dest.name, flush=True)
        return json.loads(result_path.read_text())
    torch.manual_seed(seed)
    model = load_base_model()
    loss_fn = None
    extra = {}
    if method in NORMS:
        loss_fn = single_norm_loss
        extra["norm"] = method
    elif method == "ramp":
        loss_fn = ramp_loss
        extra["gp_beta"] = 0.5
    elif method in ["alternating_distillation", "without_distillation"]:
        loss_fn = AlternatingDistillationLoss(load_base_model(),
                    distill_weight=0.5 if method == "alternating_distillation" else 0.0)
    elif method != "base":
        raise ValueError(method)
    if loss_fn is not None:
        model = adv_train(model, loss_fn, epochs=epochs, seed=seed, **extra)
        save_model(model, dest / "model.pt")
        (dest / "training_history.json").write_text(json.dumps(model.training_history, indent=2) + "\n")
        if isinstance(loss_fn, AlternatingDistillationLoss):
            (dest / "batch_statistics.json").write_text(json.dumps(loss_fn.stats) + "\n")
    sync_device()
    start = time.perf_counter()
    acc = test_model(model)  # Required 1,000 examples, 50 steps, two restarts.
    sync_device()
    result = {"method": method, "seed": seed, "epochs": epochs if method != "base" else 0,
              "accuracy_percent": acc, "evaluation_seconds": time.perf_counter() - start,
              "training_history": getattr(model, "training_history", []),
              "evaluation": {"n": 1000, "steps": 50, "restarts": 2, "batch_size": 250},
              "device": str(device), "torch": torch.__version__,
              "canonical_module_sha256": '8160d9711962fae02733f82c85013c2446649727199b37a1e74f733a9e300035'}
    result_path.write_text(json.dumps(result, indent=2) + "\n")
    print("RUN_COMPLETE", method, seed, json.dumps(acc), flush=True)
    del model, loss_fn
    if device.type == "cuda": torch.cuda.empty_cache()
    elif device.type == "mps": torch.mps.empty_cache()
    return result

def benchmark():
    model = load_base_model()
    model.eval()
    x, y = x_train[:128], y_train[:128]
    for norm in NORMS:
        sync_device()
        start = time.perf_counter()
        x_adv = attack(model, x, y, norm, steps=10)
        sync_device()
        print("BENCHMARK", norm, time.perf_counter() - start, flush=True)
        delta = x_adv - x
        ord_value = {"linf": float("inf"), "l2": 2, "l1": 1}[norm]
        max_norm = delta.flatten(1).norm(p=ord_value, dim=1).max().item()
        assert max_norm <= EPS[norm] + 1e-4, (norm, max_norm)
        assert x_adv.min() >= 0 and x_adv.max() <= 1
        assert not model.training

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--method", default="benchmark", choices=["benchmark","base",*NORMS,"ramp","alternating_distillation","without_distillation","all","strong"])
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--epochs", type=int, default=10)
    p.add_argument("--checkpoint")
    a = p.parse_args()
    torch.set_num_threads(4)
    initialize_data()
    if a.method == "benchmark":
        benchmark()
    elif a.method == "all":
        schedule = [("base",0),("linf",0),("l2",0),("l1",0),("ramp",0),("ramp",1),
                    ("alternating_distillation",0),("alternating_distillation",1),
                    ("without_distillation",0),("without_distillation",1)]
        for method, seed in schedule: execute_run(method, seed)
    elif a.method == "strong":
        if not a.checkpoint: raise ValueError("--checkpoint is required")
        model = make_model()
        model.load_state_dict(torch.load(a.checkpoint, map_location=device, weights_only=True))
        acc = test_model(model, steps=200, restarts=5)
        Path(a.checkpoint).with_name("strong_evaluation.json").write_text(
            json.dumps({"accuracy_percent": acc, "steps":200, "restarts":5, "n":1000}, indent=2) + "\n")
    else:
        execute_run(a.method, a.seed, a.epochs)



## Method in part (c)

The candidate alternates L1 and L2 adversarial updates. A frozen copy of the base model supplies clean teacher probabilities. On teacher-correct examples, the student's L-infinity-adversarial prediction is paired to this target; a small clean cross-entropy term is also retained.

`L = CE(student(x_alt), y) + 0.5 KL(teacher(x) || student(x_inf)) + 0.25 CE(student(x), y)`.

The masked KL is averaged over teacher-correct examples. This combines an alternating weak-norm update with a frozen robust anchor. It is inspired by [Adversarially Robust Distillation](https://ojs.aaai.org/index.php/AAAI/article/view/5816); no claim of global research novelty is made. The ablation removes the distillation term while preserving attack computation. The method is evaluated even if it fails to improve union accuracy.

RAMP uses the simplified homework objective, not the full published training pipeline. Its positive-cosine gradient combination and masked KL direction follow [the authors' implementation](https://github.com/uiuc-focal-lab/RAMP).


In [ ]:
torch.set_num_threads(4)
initialize_data()
benchmark()


In [ ]:
schedule = [("base",0),("linf",0),("l2",0),("l1",0),("ramp",0),("ramp",1),
            ("alternating_distillation",0),("alternating_distillation",1),
            ("without_distillation",0),("without_distillation",1)]
for method, seed in schedule:
    execute_run(method, seed)


## Stronger evaluation

Select the highest default union-accuracy checkpoint among the proposed-method runs, then evaluate it with 200 steps and five restarts. Selection uses the assignment's test set, so this is a diagnostic comparison, not an independent model-selection estimate.

In [ ]:
candidates = []
for p in (ROOT/"results/q3").glob("alternating_distillation_seed*/result.json"):
    r = json.loads(p.read_text())
    candidates.append((r["accuracy_percent"]["union"], p.parent/"model.pt"))
_, best = max(candidates, key=lambda item:(item[0],str(item[1])))
model = make_model()
model.load_state_dict(torch.load(best, weights_only=True, map_location=device))
print("Selected:", best)
strong = test_model(model, steps=200, restarts=5)
(best.parent/"strong_evaluation.json").write_text(json.dumps(strong,indent=2))


## Recorded experiment evidence

The table below is generated from saved CUDA result files. Actual epoch timings are retained in each record. Re-running the notebook produces new outputs.

In [ ]:
recorded_results = [{'method': 'alternating_distillation', 'seed': 0, 'accuracy_percent': {'clean': 80.2, 'linf': 37.2, 'l2': 62.0, 'l1': 48.7, 'union': 36.4}, 'training_history': [{'epoch': 1, 'loss': 1.4725809402954884, 'seconds': 18.05631181699937}, {'epoch': 2, 'loss': 1.3471835316755834, 'seconds': 18.04356537900094}, {'epoch': 3, 'loss': 1.2782811270310328, 'seconds': 18.08598472200174}, {'epoch': 4, 'loss': 1.222160911712891, 'seconds': 18.135218131999864}, {'epoch': 5, 'loss': 1.1533561723354535, 'seconds': 18.25283065900294}, {'epoch': 6, 'loss': 1.087333545470849, 'seconds': 18.02540448400032}, {'epoch': 7, 'loss': 1.0379883341300182, 'seconds': 18.07272578699849}, {'epoch': 8, 'loss': 0.9877251539474878, 'seconds': 18.107634222000343}, {'epoch': 9, 'loss': 0.9645295112560957, 'seconds': 18.636813417000667}, {'epoch': 10, 'loss': 0.9497256745130588, 'seconds': 18.123509456003376}]}, {'method': 'alternating_distillation', 'seed': 1, 'accuracy_percent': {'clean': 79.5, 'linf': 37.7, 'l2': 61.6, 'l1': 47.6, 'union': 36.9}, 'training_history': [{'epoch': 1, 'loss': 1.4632588441555316, 'seconds': 17.986047975002293}, {'epoch': 2, 'loss': 1.3397497511827028, 'seconds': 18.144154084999172}, {'epoch': 3, 'loss': 1.2822930667644892, 'seconds': 18.556409526001516}, {'epoch': 4, 'loss': 1.2178471615681281, 'seconds': 18.692890926002292}, {'epoch': 5, 'loss': 1.155888849344009, 'seconds': 18.399661016002938}, {'epoch': 6, 'loss': 1.0881620828921978, 'seconds': 18.02093437900112}, {'epoch': 7, 'loss': 1.0324683472132072, 'seconds': 18.10031663799964}, {'epoch': 8, 'loss': 0.9973769340759668, 'seconds': 18.277843480998854}, {'epoch': 9, 'loss': 0.9619537744766626, 'seconds': 17.9993822610013}, {'epoch': 10, 'loss': 0.9438187166666373, 'seconds': 18.061331451001024}]}, {'method': 'base', 'seed': 0, 'accuracy_percent': {'clean': 78.7, 'linf': 48.2, 'l2': 59.9, 'l1': 23.4, 'union': 23.3}, 'training_history': []}, {'method': 'l1', 'seed': 0, 'accuracy_percent': {'clean': 84.4, 'linf': 0.0, 'l2': 0.0, 'l1': 23.9, 'union': 0.0}, 'training_history': [{'epoch': 1, 'loss': 1.2303728407774217, 'seconds': 9.896297420000337}, {'epoch': 2, 'loss': 0.8321175972620646, 'seconds': 9.92943847800052}, {'epoch': 3, 'loss': 0.5769937068988116, 'seconds': 9.927232422000088}, {'epoch': 4, 'loss': 0.43241567260179764, 'seconds': 9.713480813999922}, {'epoch': 5, 'loss': 0.3287522494792938, 'seconds': 9.848718993998773}, {'epoch': 6, 'loss': 0.27909298355762774, 'seconds': 9.825802797997312}, {'epoch': 7, 'loss': 0.21120367581263566, 'seconds': 9.727331074998801}, {'epoch': 8, 'loss': 0.1748559581927764, 'seconds': 9.882571063000796}, {'epoch': 9, 'loss': 0.16750483663800436, 'seconds': 9.877698066000448}, {'epoch': 10, 'loss': 0.15210652179442918, 'seconds': 9.87788238000212}]}, {'method': 'l2', 'seed': 0, 'accuracy_percent': {'clean': 82.3, 'linf': 32.9, 'l2': 60.2, 'l1': 29.1, 'union': 25.5}, 'training_history': [{'epoch': 1, 'loss': 0.8956152483438834, 'seconds': 8.618047122999997}, {'epoch': 2, 'loss': 0.8284102013477912, 'seconds': 8.633915555001295}, {'epoch': 3, 'loss': 0.7832894531580118, 'seconds': 8.580922733999614}, {'epoch': 4, 'loss': 0.7426600899451818, 'seconds': 8.486359098998946}, {'epoch': 5, 'loss': 0.7096481109276797, 'seconds': 8.357592315998772}, {'epoch': 6, 'loss': 0.6708924839129815, 'seconds': 8.576175648999197}, {'epoch': 7, 'loss': 0.6284983784724505, 'seconds': 8.540042761000223}, {'epoch': 8, 'loss': 0.6045025147688694, 'seconds': 8.509627126997657}, {'epoch': 9, 'loss': 0.5859117538501055, 'seconds': 8.247033504998399}, {'epoch': 10, 'loss': 0.5787870597380859, 'seconds': 8.252883212000597}]}, {'method': 'linf', 'seed': 0, 'accuracy_percent': {'clean': 79.3, 'linf': 45.1, 'l2': 57.4, 'l1': 20.1, 'union': 19.9}, 'training_history': [{'epoch': 1, 'loss': 1.1996805392778838, 'seconds': 8.145633332002035}, {'epoch': 2, 'loss': 1.1801182085122817, 'seconds': 7.98967255000025}, {'epoch': 3, 'loss': 1.154173011199022, 'seconds': 8.137553625998407}, {'epoch': 4, 'loss': 1.1322301519222748, 'seconds': 8.132389139002044}, {'epoch': 5, 'loss': 1.0982567209463854, 'seconds': 8.130733148998843}, {'epoch': 6, 'loss': 1.0689236529362507, 'seconds': 8.057871684002748}, {'epoch': 7, 'loss': 1.0427338160001314, 'seconds': 8.030325196003105}, {'epoch': 8, 'loss': 1.0267897790823228, 'seconds': 8.032183334002184}, {'epoch': 9, 'loss': 1.0113051625398488, 'seconds': 8.022722015000909}, {'epoch': 10, 'loss': 1.0034914230689025, 'seconds': 8.128169920997607}]}, {'method': 'ramp', 'seed': 0, 'accuracy_percent': {'clean': 79.1, 'linf': 42.4, 'l2': 62.1, 'l1': 44.9, 'union': 40.3}, 'training_history': [{'epoch': 1, 'loss': 1.430104720286834, 'seconds': 20.325630331000866}, {'epoch': 2, 'loss': 1.334685246149699, 'seconds': 20.42144460200143}, {'epoch': 3, 'loss': 1.2993488556299455, 'seconds': 19.617710785998497}, {'epoch': 4, 'loss': 1.2682070594567518, 'seconds': 19.697469527000067}, {'epoch': 5, 'loss': 1.2530468243819017, 'seconds': 19.64779583299969}, {'epoch': 6, 'loss': 1.2226310517543402, 'seconds': 19.54258494500027}, {'epoch': 7, 'loss': 1.199038756199372, 'seconds': 19.786357498000143}, {'epoch': 8, 'loss': 1.1829441487789154, 'seconds': 19.79411779500151}, {'epoch': 9, 'loss': 1.173732837041219, 'seconds': 19.61913029900097}, {'epoch': 10, 'loss': 1.1652160493227153, 'seconds': 19.881513029002235}]}, {'method': 'ramp', 'seed': 1, 'accuracy_percent': {'clean': 78.5, 'linf': 42.2, 'l2': 62.0, 'l1': 45.5, 'union': 40.6}, 'training_history': [{'epoch': 1, 'loss': 1.4328668102239952, 'seconds': 18.613951409002766}, {'epoch': 2, 'loss': 1.3274801923678472, 'seconds': 20.199763673001144}, {'epoch': 3, 'loss': 1.296068359643985, 'seconds': 20.429730994997954}, {'epoch': 4, 'loss': 1.2759616329119756, 'seconds': 20.02735131600275}, {'epoch': 5, 'loss': 1.2533588608105977, 'seconds': 19.66356527300013}, {'epoch': 6, 'loss': 1.223206645403153, 'seconds': 19.699721587003296}, {'epoch': 7, 'loss': 1.2052296354220464, 'seconds': 19.959448124998744}, {'epoch': 8, 'loss': 1.1882728926646404, 'seconds': 20.04643563799982}, {'epoch': 9, 'loss': 1.1771985987822216, 'seconds': 20.009997446999478}, {'epoch': 10, 'loss': 1.1692111438665636, 'seconds': 20.306621971001732}]}, {'method': 'without_distillation', 'seed': 0, 'accuracy_percent': {'clean': 81.1, 'linf': 26.9, 'l2': 57.3, 'l1': 49.8, 'union': 26.6}, 'training_history': [{'epoch': 1, 'loss': 1.3031929318721478, 'seconds': 17.83226418599952}, {'epoch': 2, 'loss': 1.1561198631922405, 'seconds': 17.78570006500013}, {'epoch': 3, 'loss': 1.0835181329494867, 'seconds': 17.98262631000034}, {'epoch': 4, 'loss': 1.0215052014742143, 'seconds': 17.75539481099986}, {'epoch': 5, 'loss': 0.9551219374705584, 'seconds': 17.928785837997566}, {'epoch': 6, 'loss': 0.8852030650163308, 'seconds': 18.110111237998353}, {'epoch': 7, 'loss': 0.8314120425627782, 'seconds': 17.948450518000755}, {'epoch': 8, 'loss': 0.7826427343564156, 'seconds': 18.30342983099763}, {'epoch': 9, 'loss': 0.7602239098304358, 'seconds': 18.11151554999742}, {'epoch': 10, 'loss': 0.7474900988432077, 'seconds': 18.394367744997}]}, {'method': 'without_distillation', 'seed': 1, 'accuracy_percent': {'clean': 81.4, 'linf': 27.1, 'l2': 58.4, 'l1': 49.5, 'union': 27.1}, 'training_history': [{'epoch': 1, 'loss': 1.301071788255985, 'seconds': 18.74993897600143}, {'epoch': 2, 'loss': 1.1460398443234272, 'seconds': 18.059288430999004}, {'epoch': 3, 'loss': 1.086777191895705, 'seconds': 18.02911537000182}, {'epoch': 4, 'loss': 1.0196677629764264, 'seconds': 18.097191925000516}, {'epoch': 5, 'loss': 0.9489785035451254, 'seconds': 17.971857548000116}, {'epoch': 6, 'loss': 0.8827196405484126, 'seconds': 18.759902570000122}, {'epoch': 7, 'loss': 0.8278033695159814, 'seconds': 18.64338310400126}, {'epoch': 8, 'loss': 0.7923778119759682, 'seconds': 18.58953736699914}, {'epoch': 9, 'loss': 0.7607927513428223, 'seconds': 18.855101599998306}, {'epoch': 10, 'loss': 0.7400859380379702, 'seconds': 18.773984192997887}]}]
for r in recorded_results:
    print(r["method"], r["seed"], r["accuracy_percent"])


In [ ]:
recorded_stronger_evaluations = {'alternating_distillation_seed1_epochs10': {'accuracy_percent': {'clean': 79.5, 'linf': 37.6, 'l2': 61.5, 'l1': 45.3, 'union': 35.9}, 'steps': 200, 'restarts': 5, 'n': 1000}, 'ramp_seed1_epochs10': {'accuracy_percent': {'clean': 78.5, 'linf': 42.0, 'l2': 61.9, 'l1': 44.3, 'union': 39.7}, 'steps': 200, 'restarts': 5, 'n': 1000}}
print(json.dumps(recorded_stronger_evaluations, indent=2))
